# Boosting Lab
**Learn gradient boosted trees by running them.** This notebook follows the *Boosted Trees at the Chai Stall* guide: same chai stall, same numbers, but now you type and run the code yourself.

**How to use it**
- Run cells top to bottom with **Shift + Enter**.
- Cells marked **Your turn** are for you to change and re-run. Breaking things is part of the plan.
- Answers are at the very bottom. Try first.

| Part | What you do | Time |
|---|---|---|
| 1 | Build boosting by hand with numpy, one tree per click | 20 min |
| 2 | Prove scikit-learn does exactly the same thing | 10 min |
| 3 | Play with sliders and watch overfitting happen | 15 min |
| 4 | Race 6 models on 20,640 real California houses | 20 min |
| 5 | Look inside the model: what matters, and where it's wrong | 15 min |
| 6 | Yes/no questions (classification) | 10 min |
| 7 | Your own data, and what to try next | open-ended |

### One-time setup (Terminal on your Mac)
```bash
brew install uv libomp            # Homebrew from https://brew.sh ; libomp is needed by LightGBM
mkdir -p ~/boosting-lab && cd ~/boosting-lab
# put boosting_lab.ipynb in this folder, then:
uv init
uv add numpy pandas matplotlib scikit-learn xgboost lightgbm ipywidgets jupyterlab
uv run jupyter lab
```
Prefer VS Code? Open the `boosting-lab` folder, install the **Jupyter** extension, and pick the `.venv` kernel (top right of the notebook).

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

# the same pen colours as the guide
BLUE, RED, GREEN, INK, SOFT = "#2153c4", "#c4362c", "#127a50", "#1a2830", "#8a9aa1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
try:
    import xgboost as xgb
    print("xgboost", xgb.__version__)
except ImportError:
    xgb = None
    print("xgboost not installed. Part 4 will skip it.")
try:
    import lightgbm as lgb
    print("lightgbm", lgb.__version__)
except Exception:          # on a Mac this also catches a missing libomp
    lgb = None
    print("lightgbm not available (try: brew install libomp). Part 4 will skip it.")

---
## Part 1 · Ravi's six days, by hand
No machine-learning library yet. Just numpy and the four-step loop:
1. Start with **the average**.
2. **Residual** = actual − prediction.
3. Fit a **stump** (a one-question tree) to the residuals.
4. Add **learning rate × stump** to the prediction. Repeat.

In [ ]:
temp = np.array([15, 18, 22, 27, 31, 35], dtype=float)   # feature: temperature in °C
cups = np.array([98, 92, 77, 63, 42, 30], dtype=float)   # target: cups of chai sold
pd.DataFrame({"temp °C": temp, "cups": cups})

### A stump: one yes/no question
This function *is* the tree-learning algorithm for a stump. It tries every cut between neighbouring temperatures and keeps the one with the smallest squared error. Read it line by line.

In [ ]:
def best_stump(x, r):
    # Find the best question "x < cut?" for predicting r.
    # Returns (cut, value_if_yes, value_if_no, squared_error).
    order = np.argsort(x)
    x, r = x[order], r[order]
    best = None
    for i in range(1, len(x)):
        if x[i] == x[i - 1]:
            continue                                  # can't cut between two equal values
        yes, no = r[:i], r[i:]                        # days left / right of the cut
        sse = ((yes - yes.mean())**2).sum() + ((no - no.mean())**2).sum()
        if best is None or sse < best[3]:
            cut = (x[i - 1] + x[i]) / 2               # halfway between the neighbours
            best = (cut, yes.mean(), no.mean(), sse)
    return best

cut, yes_val, no_val, sse = best_stump(temp, cups)
print(f"Best question: colder than {cut} °C?  yes → {yes_val:.0f} cups, no → {no_val:.0f} cups"
      f"   (squared error {sse:.0f})")

In [ ]:
def predict(x, f0, stumps, lr):
    # The whole boosted model: average + lr × (sum of every stump's answer)
    p = np.full(np.shape(x), f0, dtype=float)
    for cut, yes, no in stumps:
        p = p + lr * np.where(x < cut, yes, no)
    return p

def plot_state(f0, stumps, lr, title):
    grid = np.linspace(12, 38, 600)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    if stumps:
        ax.plot(grid, predict(grid, f0, stumps[:-1], lr), "--", color=SOFT, lw=1.5, label="before this tree")
        ax.axvline(stumps[-1][0], color=GREEN, ls=":", lw=1.5, label=f"tree {len(stumps)} cuts here")
    ax.plot(grid, predict(grid, f0, stumps, lr), color=BLUE, lw=2.5, label="prediction")
    pred = predict(temp, f0, stumps, lr)
    for x, y, p in zip(temp, cups, pred):
        ax.plot([x, x], [y, p], color=RED, lw=2)                       # the red stick = residual
        ax.annotate(f"{y - p:+.0f}", (x + 0.4, (y + p) / 2), color=RED, fontsize=9, va="center")
    ax.scatter(temp, cups, color=INK, zorder=3, s=36, label="actual")
    ax.set(xlabel="temperature (°C)", ylabel="cups sold", ylim=(20, 108), title=title)
    ax.legend(fontsize=8, loc="upper right")
    plt.show()

In [ ]:
LR = 0.5                    # learning rate: add half of each tree's answer
f0 = cups.mean()            # step 1: the average
stumps = []                 # no trees yet
history = [((cups - f0)**2).sum()]
print(f"Start: predict {f0:.0f} cups for every day. Total squared error = {history[-1]:.0f}")
plot_state(f0, stumps, LR, "Start: every day gets the average")

### ▶ Run the next cell again and again
Each run adds **one** tree. Watch the red sticks shrink. Tree 1 should match the guide: *colder than 24.5 °C? yes +22, no −22*.

To start over, re-run the cell above.

In [ ]:
before = predict(temp, f0, stumps, LR)
residual = cups - before                          # step 2: what's still wrong?
cut, yes, no, _ = best_stump(temp, residual)      # step 3: a stump learns the residuals
stumps.append((cut, yes, no))                     # step 4: add it to the model
after = predict(temp, f0, stumps, LR)
history.append(((cups - after)**2).sum())

print(f"Tree {len(stumps)}: colder than {cut} °C?  yes → {yes:+.1f}, no → {no:+.1f}   (× {LR})")
display(pd.DataFrame({"temp": temp, "cups": cups, "before": before, "error": residual,
                      "+ lr × tree": after - before, "after": after, "new error": cups - after}).round(1))
print("Total squared error after each round:", [round(h) for h in history])
plot_state(f0, stumps, LR, f"After tree {len(stumps)}")

### Learning rate: big steps or small?
Same loop, packed into one function, run for three learning rates.

In [ ]:
def run_boosting(n_trees, lr):
    f0 = cups.mean()
    stumps = []
    errors = [((cups - f0)**2).sum()]
    for _ in range(n_trees):
        residual = cups - predict(temp, f0, stumps, lr)
        cut, yes, no, _ = best_stump(temp, residual)
        stumps.append((cut, yes, no))
        errors.append(((cups - predict(temp, f0, stumps, lr))**2).sum())
    return f0, stumps, errors

plt.figure(figsize=(7, 3.4))
for lr, colour in [(0.1, GREEN), (0.5, BLUE), (1.0, RED)]:
    _, _, errors = run_boosting(30, lr)
    plt.plot(errors, marker=".", color=colour, label=f"learning rate {lr}")
plt.yscale("log")
plt.xlabel("number of trees"); plt.ylabel("total squared error (log scale)")
plt.legend(); plt.show()

### Your turn 1
**a)** First *guess* what a 3-tree model (LR 0.5) predicts for **21 °C** and **33 °C**. Then check with the cell below. (The guide's pencil exercise has the same question.)

**b)** With learning rate 0.1, how many trees does it take to get the total squared error below 300? With 1.0? Use `run_boosting`.

**c)** Look at the chart. You'd expect the biggest learning rate (1.0) to win. Does it? Write down a guess for why, then check the answers.

In [ ]:
# a) a 3-tree model, LR 0.5
f0_3, stumps_3, _ = run_boosting(3, 0.5)
print(predict(np.array([21.0, 33.0]), f0_3, stumps_3, 0.5))

# b) your code here: hint, look at where `errors` first drops below 300

---
## Part 2 · The same thing with scikit-learn
Real libraries run the exact loop you just wrote. Here's the 10-line version from the guide, checked against scikit-learn's own `GradientBoostingRegressor`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import GradientBoostingRegressor

X6 = temp.reshape(-1, 1)        # scikit-learn wants a 2-D table: rows × features

def boost(X, y, n_trees=100, lr=0.1, depth=2):
    f0 = y.mean()                                   # step 1
    pred = np.full(len(y), f0)
    trees = []
    for _ in range(n_trees):
        residual = y - pred                         # step 2
        tree = DecisionTreeRegressor(max_depth=depth).fit(X, residual)   # step 3
        pred += lr * tree.predict(X)                # step 4
        trees.append(tree)
    return f0, trees

def boost_predict(f0, trees, X, lr=0.1):
    return f0 + lr * sum(t.predict(X) for t in trees)

new_days = np.array([[21.0], [33.0]])
f0_s, trees = boost(X6, cups, n_trees=3, lr=0.5, depth=1)
gbr = GradientBoostingRegressor(n_estimators=3, learning_rate=0.5, max_depth=1).fit(X6, cups)

print("our 10-line version:", boost_predict(f0_s, trees, new_days, lr=0.5))
print("scikit-learn:       ", gbr.predict(new_days))

Same answers. Now look at scikit-learn's three trees. It writes the question as `temp <= 24.5` (True = yes), which is the same split on these days. `value` is what each leaf adds *before* the learning rate is applied: +22 and −22, just like the guide.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for i, ax in enumerate(axes):
    plot_tree(gbr.estimators_[i, 0], feature_names=["temp"], filled=True, impurity=False,
              ax=ax, fontsize=9, precision=1)
    ax.set_title(f"tree {i + 1}")
plt.show()

---
## Part 3 · The playground, in code
Ravi's whole month: **30 days to learn from**, **15 days the model never sees**. The unseen days tell you how good the model really is.

In [ ]:
rng = np.random.default_rng(17)

def chai_days(n):
    t = rng.uniform(12, 38, n).round(1)
    c = 36 + 60 / (1 + np.exp((t - 25) / 2.6)) + rng.normal(0, 6, n)   # true pattern + noise
    return t.reshape(-1, 1), c.round()

Xm_train, ym_train = chai_days(30)
Xm_test, ym_test = chai_days(15)

plt.figure(figsize=(7, 3.4))
plt.scatter(Xm_train, ym_train, color=INK, label="training days (30)")
plt.scatter(Xm_test, ym_test, facecolors="none", edgecolors=INK, label="new days (15)")
plt.xlabel("temperature (°C)"); plt.ylabel("cups sold"); plt.legend(); plt.show()

In [ ]:
def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p))**2)))   # "typically off by"

def show_model(n_trees=35, learning_rate=0.1, depth=1):
    m = GradientBoostingRegressor(n_estimators=n_trees, learning_rate=learning_rate,
                                  max_depth=depth, random_state=0).fit(Xm_train, ym_train)
    train_curve = [rmse(ym_train, p) for p in m.staged_predict(Xm_train)]   # error after each tree
    test_curve = [rmse(ym_test, p) for p in m.staged_predict(Xm_test)]
    grid = np.linspace(10, 40, 600).reshape(-1, 1)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
    a1.scatter(Xm_train, ym_train, color=INK, s=22)
    a1.scatter(Xm_test, ym_test, facecolors="none", edgecolors=INK, s=22)
    a1.plot(grid, m.predict(grid), color=BLUE, lw=2.2)
    a1.set(title=f"{n_trees} trees · learning rate {learning_rate} · depth {depth}",
           xlabel="temperature (°C)", ylabel="cups sold")
    rounds = np.arange(1, n_trees + 1)
    a2.plot(rounds, train_curve, color=BLUE, label=f"training days: ±{train_curve[-1]:.1f}")
    a2.plot(rounds, test_curve, color=RED, label=f"new days: ±{test_curve[-1]:.1f}")
    best = int(np.argmin(test_curve))
    a2.scatter(best + 1, test_curve[best], color=RED, zorder=3)
    a2.annotate(f"best: {best + 1} trees", (best + 1, test_curve[best]),
                textcoords="offset points", xytext=(6, 10), color=RED)
    a2.set(xlabel="number of trees", ylabel="typically off by (cups)", ylim=(0, None))
    a2.legend()
    plt.show()

show_model(35, 0.1, 1)      # careful learner
show_model(200, 0.3, 3)     # overeager: watch the red curve climb back up

### ▶ Sliders
Drag them and the model retrains instantly. (If you see text instead of sliders, `ipywidgets` isn't installed in this kernel.)

In [ ]:
from ipywidgets import interact, IntSlider, FloatLogSlider

interact(show_model,
         n_trees=IntSlider(value=35, min=1, max=300, continuous_update=False),
         learning_rate=FloatLogSlider(value=0.1, base=10, min=-2, max=0, step=0.05, continuous_update=False),
         depth=IntSlider(value=1, min=1, max=5, continuous_update=False));

### Your turn 2
**a)** Find the slider setting with the lowest **new-day** error. Is it the one with the lowest *training* error?

**b)** Edit `show_model`: add `subsample=0.7` inside `GradientBoostingRegressor(...)`. Now each tree sees a random 70% of the days. Re-run. What changes in the curves?

**c)** In the "overeager" chart, training error goes almost to zero. Why can depth-3 trees do that, when stumps can't?

---
## Part 4 · Real data: California house prices
20,640 neighbourhoods from the 1990 US census. 8 features (income, house age, rooms, location…). Target: median house value in **$100,000s**. It downloads once and is cached on your Mac.

We split the rows three ways:
- **train** (60%): the models learn from these
- **validation** (20%): used only to decide *when to stop adding trees*
- **test** (20%): the final exam, looked at once at the end

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
display(X.head())

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42)
print(len(X_train), "train /", len(X_val), "validation /", len(X_test), "test")

### The race
Same data, six models. Boosted models get a validation set and stop when it stops improving (**early stopping**). On an M4 this whole cell takes well under a minute.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

results = []
def race(name, model, **fit_args):
    t0 = time.perf_counter()
    model.fit(X_train, y_train, **fit_args)
    secs = time.perf_counter() - t0
    err = rmse(y_test, model.predict(X_test))
    results.append({"model": name, "test error ($100k)": round(err, 3), "seconds": round(secs, 2)})
    print(f"{name:<30} off by ±{err:.3f}   ({secs:.1f} s)")

race("Just the average", DummyRegressor())
race("Linear regression", LinearRegression())
race("Random forest, 300 trees", RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0))

hgb = HistGradientBoostingRegressor(max_iter=2000, learning_rate=0.05, early_stopping=True,
                                    n_iter_no_change=50, random_state=0)
race("sklearn HistGradientBoosting", hgb)       # uses 10% of train internally for early stopping

if xgb:
    xgbm = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=6, subsample=0.8,
                            colsample_bytree=0.8, early_stopping_rounds=100, eval_metric="rmse")
    race("XGBoost", xgbm, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)

if lgb:
    lgbm = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=63, subsample=0.8,
                             subsample_freq=1, colsample_bytree=0.8, verbose=-1)
    race("LightGBM", lgbm, eval_X=X_val, eval_y=y_val,
         callbacks=[lgb.early_stopping(100, verbose=False)])

In [ ]:
board = pd.DataFrame(results).set_index("model")
display(board)
colours = [SOFT, SOFT, GREEN] + [BLUE] * (len(board) - 3)
board["test error ($100k)"].plot.barh(color=colours, figsize=(7, 3.2))
plt.gca().invert_yaxis()
plt.xlabel("typical error on test houses ($100k). Shorter is better.")
plt.show()

### Early stopping, as a picture
XGBoost kept a log of its error after every tree. The green line is where validation error was lowest. Every tree after that was only memorizing.

In [ ]:
if xgb:
    log = xgbm.evals_result()
    train_err, val_err = log["validation_0"]["rmse"], log["validation_1"]["rmse"]
    plt.figure(figsize=(7, 3.4))
    plt.plot(train_err, color=BLUE, label="training rows")
    plt.plot(val_err, color=RED, label="validation rows")
    plt.axvline(xgbm.best_iteration, color=GREEN, ls=":", label=f"best: tree {xgbm.best_iteration + 1}")
    plt.xlabel("number of trees"); plt.ylabel("RMSE ($100k)"); plt.legend(); plt.show()
    print("trees trained:", len(train_err), "| trees used for predictions:", xgbm.best_iteration + 1)

### Your turn 3: beat the leaderboard
Copy the XGBoost lines into the cell below and change one knob at a time:
- `learning_rate`: 0.02, then 0.1
- `max_depth`: 4, then 8
- `subsample`: 1.0, then 0.6

Judge each try by **validation** error (`xgbm.best_score`), not test error. Why? If you keep peeking at the test set while tuning, it stops being a fair final exam.

In [ ]:
# your experiments here

---
## Part 5 · Look inside the model
A boosted model is hundreds of trees, so you don't read it tree by tree. Instead you ask it questions.

**Which columns matter?** Shuffle one column and see how much worse the model gets.

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

imp = permutation_importance(hgb, X_test, y_test, n_repeats=5, random_state=0, n_jobs=-1)
order = imp.importances_mean.argsort()
plt.figure(figsize=(7, 3.4))
plt.barh(X.columns[order], imp.importances_mean[order], color=GREEN)
plt.xlabel("how much worse the model gets when this column is shuffled"); plt.show()

**How does each column push the prediction?** A *partial dependence* plot changes one column while holding the rest, and draws the average prediction. Notice the staircase shapes: those are trees.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.4))
PartialDependenceDisplay.from_estimator(hgb, X_test, ["MedInc", "HouseAge", "AveOccup"], ax=ax,
                                        line_kw={"color": BLUE})
plt.show()

**Where is it wrong?** Every test house on a map of California. Left: what the model predicts. Right: its mistakes.

In [ ]:
pred_test = hgb.predict(X_test)
error = y_test - pred_test                       # actual − predicted, like the red sticks
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True, sharey=True)
s = axes[0].scatter(X_test.Longitude, X_test.Latitude, c=pred_test, cmap="viridis", s=4)
fig.colorbar(s, ax=axes[0], label="$100k"); axes[0].set_title("predicted house value")
s = axes[1].scatter(X_test.Longitude, X_test.Latitude, c=error, cmap="RdBu", vmin=-1.5, vmax=1.5, s=4)
fig.colorbar(s, ax=axes[1], label="actual − predicted")
axes[1].set_title("blue: model guessed too low · red: too high")
for a in axes:
    a.set(xlabel="longitude", ylabel="latitude"); a.grid(False)
plt.show()

Want to explain *single* predictions ("why is this house $450k?")? Install SHAP with `uv add shap`, then:
```python
import shap
explainer = shap.TreeExplainer(xgbm)
shap.plots.waterfall(explainer(X_test.iloc[:1])[0])
```

---
## Part 6 · Yes/no questions (classification)
Same loop. The model adds up a score, a sigmoid turns it into a probability, and each tree chases *actual (0 or 1) − predicted probability*. This dataset ships with scikit-learn: 569 tumours, 30 measurements, benign or malignant.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)

clf = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, random_state=0).fit(Xc_train, yc_train)
proba = clf.predict_proba(Xc_test)[:, 1]          # probability of "benign"
print(f"accuracy {accuracy_score(yc_test, proba > 0.5):.3f}   ROC AUC {roc_auc_score(yc_test, proba):.3f}")

plt.figure(figsize=(7, 3))
plt.hist(proba[yc_test == 1], bins=20, range=(0, 1), color=GREEN, alpha=0.8, label="actually benign")
plt.hist(proba[yc_test == 0], bins=20, range=(0, 1), color=RED, alpha=0.8, label="actually malignant")
plt.xlabel("predicted probability of benign"); plt.ylabel("tumours"); plt.legend(); plt.show()

---
## Part 7 · Your own data
Fill in the file name and the column you want to predict, then uncomment. `HistGradientBoostingRegressor` handles missing values on its own, and text columns once they're turned into categories.

In [ ]:
# df = pd.read_csv("my_data.csv")
# target = "price"                                   # the column to predict
# X_own = df.drop(columns=[target])
# for col in X_own.select_dtypes("object"):
#     X_own[col] = X_own[col].astype("category")    # text → categories
# y_own = df[target]
#
# Xo_train, Xo_test, yo_train, yo_test = train_test_split(X_own, y_own, test_size=0.2, random_state=0)
# model = HistGradientBoostingRegressor(categorical_features="from_dtype", early_stopping=True,
#                                       random_state=0).fit(Xo_train, yo_train)
# print("off by ±", rmse(yo_test, model.predict(Xo_test)), "| the average alone:", rmse(yo_test, [yo_train.mean()] * len(yo_test)))

### What to try next
1. **Bike sharing demand** is Ravi's chai stall at city scale: weather and hour of day → bikes rented. Load it with
   `from sklearn.datasets import fetch_openml; bikes = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True)`.
2. **Kaggle "House Prices: Advanced Regression Techniques"**: messy, real tabular data, the classic place to practise boosting.
3. **Kaggle "Titanic"**: the classic yes/no problem for Part 6's classifier.
4. When you're comfortable, try **CatBoost** (`uv add catboost`). It's especially good with text categories.

---
## Answers (try first!)

**Your turn 1**
- a) **80** cups at 21 °C and **43** cups at 33 °C: 67 + 0.5 × (22 + 10 − 6) and 67 + 0.5 × (−22 − 20 − 6).
- b) Count with `next(i for i, e in enumerate(errors) if e < 300)`. Learning rate 0.1 needs **18** trees, 0.5 needs **3**, and 1.0 needs **6**.
- c) Not early on. Learning rate 1.0 starts fastest (error 792 after one tree), then crawls: 522, 468, 360… Each tree applies its rough two-group answer in full, overshoots some days, and the next trees spend effort undoing that. Half-size steps leave room for the next tree's better question. After about 15 trees, 1.0 does pull ahead, but only on the six days it's learning from. Driving *training* error to zero is easy: you can memorize. What matters is error on new days, and Part 3 shows that's where small steps win.

**Your turn 2**
- a) The lowest new-day error comes from a gentle setting (small learning rate, shallow trees, a few dozen trees). The lowest *training* error comes from the overeager settings. They are not the same thing, and that gap is overfitting.
- b) The curves get slightly wiggly (each tree sees different days), and new-day error often ends a little lower. This is called *stochastic gradient boosting*: randomness that fights overfitting.
- c) A depth-3 tree has up to 8 leaves, so it can carve out tiny groups of 1 to 3 days. After enough rounds the model has a leaf for almost every training day, noise included. A stump can only split the days into two groups each round.

**Your turn 3**
- A smaller learning rate with more trees usually helps a little and takes longer. Very deep trees tend to overfit sooner (watch `best_iteration` drop). Expect the three boosting libraries to land close to each other and well ahead of linear regression: tuning gives small gains, while the jump from "no boosting" to "boosting" is the big one.